# Notebook 02: Demand Forecasting Demo
**AAIRM** — Agentic AI Inventory Replenishment and Management  
Paper: Syed et al. (2025), *Agentic Commerce*, Frontiers in [Journal].

Demonstrates C1 DemandForecastingAgent with NaiveForecaster and compares to actual demand.

In [ ]:
import syssys.path.insert(0, '..')from aairm.utils.seed import set_global_seedfrom aairm.utils.config import AAIRMConfigfrom aairm.simulation.sku_catalog import SKUCatalogfrom aairm.simulation.demand_generator import DemandGeneratorfrom aairm.models.forecasting.naive_forecaster import NaiveForecasterimport numpy as np, pandas as pd, matplotlib.pyplot as pltset_global_seed(42)print('OK')

## 1. Generate Sample SKU Data

In [ ]:
catalog = SKUCatalog(n_skus=50, seed=42)gen = DemandGenerator(catalog, n_days=90, seed=42)sku_id = catalog.skus_by_category('grocery')[0]history = gen.get_history(sku_id, 60, 60)print(f'SKU: {sku_id}  History shape: {history.shape}')

## 2. Naive Forecaster (7-day horizon)

In [ ]:
forecaster = NaiveForecaster(window=7)ctx = {'rolling_7d_mean': float(history[-7:].mean()), 'rolling_7d_std': float(history[-7:].std()+1e-8)}result = forecaster.predict(sku_id, history, ctx, horizon=7)print('Forecast result:')for k, v in result.items():    print(f'  {k}: {v}')

## 3. Visualise Forecast vs Actual

In [ ]:
actual_next7 = gen.get_history(sku_id, 7, 67)fig, ax = plt.subplots(figsize=(10, 4))ax.plot(range(60), history, label='History', color='steelblue')ax.axhline(result['mean']/7, color='orange', linestyle='--', label=f'Forecast mean/day')ax.fill_between(range(60, 67), result['p10']/7, result['p90']/7, alpha=0.3, color='orange', label='P10–P90')ax.plot(range(60, 67), actual_next7, color='green', marker='o', label='Actual')ax.legend(); ax.set_title(f'Demand Forecast for {sku_id}')plt.tight_layout(); plt.show()